In [8]:
import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score

In [9]:
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

print(train.shape)
print(test.shape)

(577347, 12)
(247435, 11)


In [10]:
X = train.drop(columns=["id", "class"])
y = train["class"]

X_test = test.drop(columns=["id"])

# Color indices
X["u_g"] = X["u"] - X["g"]
X["g_r"] = X["g"] - X["r"]
X["r_i"] = X["r"] - X["i"]
X["i_z"] = X["i"] - X["z"]

X["u_r"] = X["u"] - X["r"]
X["g_i"] = X["g"] - X["i"]
X["r_z"] = X["r"] - X["z"]

X_test["u_g"] = X_test["u"] - X_test["g"]
X_test["g_r"] = X_test["g"] - X_test["r"]
X_test["r_i"] = X_test["r"] - X_test["i"]
X_test["i_z"] = X_test["i"] - X_test["z"]

X_test["u_r"] = X_test["u"] - X_test["r"]
X_test["g_i"] = X_test["g"] - X_test["i"]
X_test["r_z"] = X_test["r"] - X_test["z"]

In [ ]:
# # Ratios
# X["u_over_g"] = X["u"] / (X["g"] + 1e-6)
# X["g_over_r"] = X["g"] / (X["r"] + 1e-6)
# X["r_over_i"] = X["r"] / (X["i"] + 1e-6)
# X["i_over_z"] = X["i"] / (X["z"] + 1e-6)

# X_test["u_over_g"] = X_test["u"] / (X_test["g"] + 1e-6)
# X_test["g_over_r"] = X_test["g"] / (X_test["r"] + 1e-6)
# X_test["r_over_i"] = X_test["r"] / (X_test["i"] + 1e-6)
# X_test["i_over_z"] = X_test["i"] / (X_test["z"] + 1e-6)

In [ ]:
# X["redshift_u"] = X["redshift"] * X["u"]
# X["redshift_g"] = X["redshift"] * X["g"]
# X["redshift_r"] = X["redshift"] * X["r"]

# X_test["redshift_u"] = X_test["redshift"] * X_test["u"]
# X_test["redshift_g"] = X_test["redshift"] * X_test["g"]
# X_test["redshift_r"] = X_test["redshift"] * X_test["r"]

In [11]:
X = X.drop(
    columns=["spectral_type", "galaxy_population"]
)

X_test = X_test.drop(
    columns=["spectral_type", "galaxy_population"]
)

In [12]:
def objective(trial):

    learning_rate = trial.suggest_float(
    "learning_rate",
    0.01,
    0.2,
    log=True
    )

    max_depth = trial.suggest_int(
        "max_depth",
        4,
        12
    )

    num_leaves = trial.suggest_int(
        "num_leaves",
        20,
        150
    )

    feature_fraction = trial.suggest_float(
        "feature_fraction",
        0.6,
        1.0
    )

    bagging_fraction = trial.suggest_float(
        "bagging_fraction",
        0.6,
        1.0
    )

    min_child_samples = trial.suggest_int(
        "min_child_samples",
        5,
        100
    )


    skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
    )

    scores = []

    for train_idx, valid_idx in skf.split(X, y):

        X_train = X.iloc[train_idx]
        X_valid = X.iloc[valid_idx]

        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]

        model = LGBMClassifier(
        learning_rate=learning_rate,
        max_depth=max_depth,
        num_leaves=num_leaves,
        feature_fraction=feature_fraction,
        bagging_fraction=bagging_fraction,
        min_child_samples=min_child_samples,
        n_estimators=1000,
        random_state=42,
        verbosity=-1
    )

        model.fit(X_train, y_train)

        preds = model.predict(X_valid)

        score = balanced_accuracy_score(
            y_valid,
            preds
        )

        scores.append(score)

    return np.mean(scores)

In [13]:
study = optuna.create_study(
    direction="maximize"
)

study.optimize(
    objective,
    n_trials=30
)

[I 2026-06-19 14:53:21,282] A new study created in memory with name: no-name-3473ca0b-1e8b-4016-b93e-d8f0f82bf95c
[I 2026-06-19 14:55:33,169] Trial 0 finished with value: 0.954392855012593 and parameters: {'learning_rate': 0.029577717575816096, 'max_depth': 5, 'num_leaves': 65, 'feature_fraction': 0.9972923806671732, 'bagging_fraction': 0.8562139932155282, 'min_child_samples': 80}. Best is trial 0 with value: 0.954392855012593.
[I 2026-06-19 14:58:55,395] Trial 1 finished with value: 0.9563803063990161 and parameters: {'learning_rate': 0.02256802721786793, 'max_depth': 9, 'num_leaves': 119, 'feature_fraction': 0.6911062529580374, 'bagging_fraction': 0.6216341981552874, 'min_child_samples': 64}. Best is trial 1 with value: 0.9563803063990161.
[I 2026-06-19 15:01:10,180] Trial 2 finished with value: 0.9544055453838167 and parameters: {'learning_rate': 0.01976204248107068, 'max_depth': 9, 'num_leaves': 29, 'feature_fraction': 0.9549387812410308, 'bagging_fraction': 0.8648666902098601, 'mi

In [14]:
print(study.best_value)

print(study.best_params)

0.9566250776113007
{'learning_rate': 0.0439451957175871, 'max_depth': 11, 'num_leaves': 103, 'feature_fraction': 0.651806792136691, 'bagging_fraction': 0.7084411760529633, 'min_child_samples': 11}


In [15]:
optuna_model_v1 = LGBMClassifier(
    learning_rate=0.0439451957175871,
    max_depth=11,
    num_leaves=103,
    feature_fraction=0.651806792136691,
    bagging_fraction=0.7084411760529633,
    min_child_samples=11,
    n_estimators=1000,
    random_state=42,
    verbosity=-1
)

optuna_model_v1.fit(X, y)

,num_leaves,103
,max_depth,11
,learning_rate,0.0439451957175871
,n_estimators,1000
,min_child_samples,11
,random_state,42
,feature_fraction,0.651806792136691
,bagging_fraction,0.7084411760529633
,verbosity,-1
,boosting_type,'gbdt'
,subsample_for_bin,200000


In [16]:
import joblib

joblib.dump(optuna_model_v1, "../models/lightgbm_optuna_v1.pkl")

['../models/lightgbm_optuna_v1.pkl']

In [17]:
preds = optuna_model_v1.predict(X_test)

In [18]:
submission = pd.DataFrame({
    "id": test["id"],
    "class": preds
})

submission.to_csv(
    "../submissions/lightgbm_optuna_v1.csv",
    index=False
)